# Advanced Problems with Solutions: Python 3.6+ Preserved Order of `**kwargs`

This notebook contains advanced practice problems with complete solutions.

Topic focus:

- Preserved order of keyword arguments collected by `**kwargs`
- Practical uses of ordered keyword arguments
- Dynamic `namedtuple` factories
- Ordered validation and serialization
- Decorators that preserve keyword order
- API design patterns that depend on keyword order

Python 3.6 introduced preserved order for `**kwargs` as part of PEP 468. In modern Python versions, regular dictionaries also preserve insertion order as a language guarantee, but this notebook focuses specifically on the practical implications of ordered `**kwargs`.


## Setup

In [1]:
from collections import namedtuple, OrderedDict
from functools import wraps
import inspect
import json
import keyword


---

## Problem 1: Build a Defaulted Named Tuple Factory

Write a function called `defaulted_namedtuple` that accepts a class name and ordered keyword arguments.

The keyword argument names should become the named tuple field names.

The keyword argument values should become default values.

Example:

```python
Point = defaulted_namedtuple('Point', x=0, y=0, label='origin')
p = Point(10)
```

Expected result:

```python
Point(x=10, y=0, label='origin')
```

Requirements:

1. Preserve the order of fields exactly as provided.
2. Support partial positional construction.
3. Reject calls with no fields.
4. Return a real `namedtuple` subclass.


### Solution

In [2]:
def defaulted_namedtuple(class_name, **fields):
    if not fields:
        raise ValueError('At least one field must be provided.')

    Struct = namedtuple(class_name, fields.keys())
    Struct.__new__.__defaults__ = tuple(fields.values())
    return Struct


In [3]:
Point = defaulted_namedtuple('Point', x=0, y=0, label='origin')

p1 = Point()
p2 = Point(10)
p3 = Point(10, 20)
p4 = Point(10, 20, 'A')

print(Point._fields)
print(p1)
print(p2)
print(p3)
print(p4)


('x', 'y', 'label')
Point(x=0, y=0, label='origin')
Point(x=10, y=0, label='origin')
Point(x=10, y=20, label='origin')
Point(x=10, y=20, label='A')


### Explanation

`fields.keys()` preserves the order in which the keyword arguments were provided.

`fields.values()` uses the same order, so the default values line up correctly with the field names.

Without ordered `**kwargs`, this factory could accidentally assign defaults to the wrong fields.


---

## Problem 2: Add Validation to the Named Tuple Factory

Improve `defaulted_namedtuple` by adding validation.

Create a function called `validated_defaulted_namedtuple`.

Requirements:

1. Field names must be valid Python identifiers.
2. Field names may not be Python keywords.
3. Field names may not start with an underscore.
4. The order of fields must be preserved.
5. Error messages should identify the first invalid field in user-provided order.


### Solution

In [4]:
def validated_defaulted_namedtuple(class_name, **fields):
    if not fields:
        raise ValueError('At least one field must be provided.')

    for field_name in fields:
        if not field_name.isidentifier():
            raise ValueError(f'{field_name!r} is not a valid Python identifier.')
        if keyword.iskeyword(field_name):
            raise ValueError(f'{field_name!r} is a Python keyword.')
        if field_name.startswith('_'):
            raise ValueError(f'{field_name!r} may not start with an underscore.')

    Struct = namedtuple(class_name, fields.keys())
    Struct.__new__.__defaults__ = tuple(fields.values())
    return Struct


In [5]:
UserRecord = validated_defaulted_namedtuple(
    'UserRecord',
    user_id=None,
    username='',
    email='',
    is_active=True
)

record = UserRecord(user_id=1001, username='simeon')

print(UserRecord._fields)
print(record)


('user_id', 'username', 'email', 'is_active')
UserRecord(user_id=1001, username='simeon', email='', is_active=True)


In [6]:
try:
    Bad = validated_defaulted_namedtuple('Bad', **{'class': 'invalid'})
except ValueError as ex:
    print(type(ex).__name__, ex)

try:
    Bad = validated_defaulted_namedtuple('Bad', _private=123)
except ValueError as ex:
    print(type(ex).__name__, ex)


ValueError 'class' is a Python keyword.
ValueError '_private' may not start with an underscore.


### Explanation

The validation loop processes `fields` in the same order in which the user passed the keyword arguments.

This matters when producing helpful error messages, building schemas, or reporting the first invalid field in a predictable way.


---

## Problem 3: Create an Ordered SQL `INSERT` Builder

Write a function called `build_insert`.

It should accept a table name and ordered keyword arguments representing column-value pairs.

Example:

```python
sql, params = build_insert('users', id=1, username='alice', active=True)
```

Expected result:

```python
sql == 'INSERT INTO users (id, username, active) VALUES (?, ?, ?)'
params == (1, 'alice', True)
```

Requirements:

1. Preserve the order of the columns.
2. Return parameterized SQL using question-mark placeholders.
3. Return the values as a tuple in the same order as the columns.
4. Reject an empty column list.
5. Do not manually interpolate values into the SQL string.


### Solution

In [7]:
def build_insert(table_name, **columns):
    if not columns:
        raise ValueError('At least one column must be provided.')

    column_names = ', '.join(columns.keys())
    placeholders = ', '.join('?' for _ in columns)
    values = tuple(columns.values())

    sql = f'INSERT INTO {table_name} ({column_names}) VALUES ({placeholders})'
    return sql, values


In [8]:
sql, params = build_insert(
    'users',
    id=1,
    username='alice',
    email='alice@example.com',
    active=True
)

print(sql)
print(params)


INSERT INTO users (id, username, email, active) VALUES (?, ?, ?, ?)
(1, 'alice', 'alice@example.com', True)


### Explanation

The SQL column list and the parameter tuple are both produced from the same ordered `columns` dictionary.

Because `**kwargs` order is preserved, the first placeholder corresponds to the first keyword argument, the second placeholder corresponds to the second keyword argument, and so on.


---

## Problem 4: Build an Ordered Schema Definition DSL

Create a function called `schema` that accepts ordered field definitions through `**kwargs`.

Each keyword argument maps a field name to a Python type.

Example:

```python
UserSchema = schema(id=int, username=str, active=bool)
```

The function should return a dictionary with:

```python
{
    'fields': ('id', 'username', 'active'),
    'types': (int, str, bool),
    'required': ('id', 'username', 'active')
}
```

Requirements:

1. Preserve field order.
2. Validate that each value is a type object.
3. Produce predictable error messages using field order.
4. Return immutable tuples instead of lists.


### Solution

In [9]:
def schema(**field_types):
    if not field_types:
        raise ValueError('A schema must contain at least one field.')

    for field_name, field_type in field_types.items():
        if not isinstance(field_type, type):
            raise TypeError(
                f'Field {field_name!r} expected a type object, '
                f'but received {field_type!r}.'
            )

    return {
        'fields': tuple(field_types.keys()),
        'types': tuple(field_types.values()),
        'required': tuple(field_types.keys())
    }


In [10]:
UserSchema = schema(
    id=int,
    username=str,
    email=str,
    active=bool
)

print(UserSchema)

try:
    BadSchema = schema(
        id=int,
        username=str,
        active='bool should not be a string',
        created_at=str
    )
except TypeError as ex:
    print(type(ex).__name__, ex)


{'fields': ('id', 'username', 'email', 'active'), 'types': (<class 'int'>, <class 'str'>, <class 'str'>, <class 'bool'>), 'required': ('id', 'username', 'email', 'active')}
TypeError Field 'active' expected a type object, but received 'bool should not be a string'.


### Explanation

The function validates fields in user-provided order.

This is useful in schema systems because field order often affects generated documentation, forms, CSV exports, table displays, and validation reports.


---

## Problem 5: Validate Records Against an Ordered Schema

Using the `schema` function from Problem 4, write a function called `validate_record`.

It should accept a schema definition and keyword arguments representing data.

Requirements:

1. Validate fields in schema order, not input order.
2. Report missing fields in schema order.
3. Report type errors in schema order.
4. Reject unexpected fields.
5. Return a normalized `OrderedDict`.


### Solution

In [11]:
def validate_record(schema_def, **data):
    expected_fields = schema_def['fields']
    expected_types = schema_def['types']
    expected_field_set = set(expected_fields)
    actual_field_set = set(data.keys())

    unexpected = actual_field_set - expected_field_set
    if unexpected:
        ordered_unexpected = [name for name in data if name in unexpected]
        raise ValueError(f'Unexpected field(s): {ordered_unexpected}')

    missing = [name for name in expected_fields if name not in data]
    if missing:
        raise ValueError(f'Missing required field(s): {missing}')

    normalized = OrderedDict()

    for field_name, expected_type in zip(expected_fields, expected_types):
        value = data[field_name]
        if not isinstance(value, expected_type):
            raise TypeError(
                f'Field {field_name!r} expected {expected_type.__name__}, '
                f'but received {type(value).__name__}.'
            )
        normalized[field_name] = value

    return normalized


In [12]:
UserSchema = schema(id=int, username=str, email=str, active=bool)

record = validate_record(
    UserSchema,
    active=True,
    email='alice@example.com',
    username='alice',
    id=1
)

print(record)
print(list(record.keys()))


OrderedDict({'id': 1, 'username': 'alice', 'email': 'alice@example.com', 'active': True})
['id', 'username', 'email', 'active']


### Explanation

Even though the input keyword arguments are passed in a different order, the normalized result follows the schema order.

Ordered `**kwargs` makes the input order available, but robust validation systems often choose one specific order for output: usually schema order.


---

## Problem 6: Build a Decorator That Logs Keyword Argument Order

Write a decorator called `log_kwargs_order`.

It should wrap a function and print the keyword arguments in the exact order in which the caller provided them.

Requirements:

1. Preserve the original function metadata using `functools.wraps`.
2. Print keyword argument names in call order.
3. Return the wrapped function's original result.
4. Work with positional and keyword arguments.


### Solution

In [13]:
def log_kwargs_order(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print('Keyword argument order:', tuple(kwargs.keys()))
        return func(*args, **kwargs)
    return wrapper


In [14]:
@log_kwargs_order
def configure_service(name, **settings):
    return {
        'name': name,
        'settings': settings
    }

result = configure_service(
    'email',
    host='smtp.example.com',
    port=587,
    use_tls=True,
    timeout=30
)

print(result)
print(configure_service.__name__)


Keyword argument order: ('host', 'port', 'use_tls', 'timeout')
{'name': 'email', 'settings': {'host': 'smtp.example.com', 'port': 587, 'use_tls': True, 'timeout': 30}}
configure_service


### Explanation

The wrapper receives `**kwargs` in call order.

This is especially useful for debugging configuration APIs where the order of options may reveal how a user expects configuration to be applied.


---

## Problem 7: Ordered Pipeline Builder

Create a function called `pipeline`.

It should accept ordered keyword arguments where each value is a callable step.

The returned function should apply each step in the order provided.

Requirements:

1. Preserve step order.
2. Validate that every step is callable.
3. Return a function.
4. Attach a `steps` attribute to the returned function containing the ordered step names.


### Solution

In [15]:
def pipeline(**steps):
    if not steps:
        raise ValueError('At least one pipeline step is required.')

    for name, step in steps.items():
        if not callable(step):
            raise TypeError(f'Pipeline step {name!r} is not callable.')

    def composed(value):
        result = value
        for step in steps.values():
            result = step(result)
        return result

    composed.steps = tuple(steps.keys())
    return composed


In [16]:
clean_text = pipeline(
    strip=str.strip,
    lower=str.lower,
    title=str.title
)

print(clean_text('  hello world  '))
print(clean_text.steps)


Hello World
('strip', 'lower', 'title')


### Explanation

This pattern is a clean example of why ordered `**kwargs` matters.

The user can define a readable transformation pipeline without passing a list of pairs or an `OrderedDict`.


---

## Problem 8: Ordered CLI Command Builder

Write a function called `command`.

It should build a command-line string from a base command and ordered keyword arguments.

Rules:

1. Single-letter options should use one dash, e.g. `m='http.server'` becomes `-m http.server`.
2. Multi-letter options should use two dashes, e.g. `directory='public'` becomes `--directory public`.
3. Underscores in option names should become hyphens.
4. Boolean `True` means include the flag without a value.
5. Boolean `False` or `None` means omit the option.
6. Preserve option order.


### Solution

In [17]:
def command(base, **options):
    parts = [base]

    for option_name, value in options.items():
        if value is False or value is None:
            continue

        normalized_name = option_name.replace('_', '-')
        prefix = '-' if len(normalized_name) == 1 else '--'
        flag = prefix + normalized_name

        if value is True:
            parts.append(flag)
        else:
            parts.append(flag)
            parts.append(str(value))

    return ' '.join(parts)


In [18]:
cmd = command(
    'python',
    m='http.server',
    bind='127.0.0.1',
    directory='public',
    verbose=True,
    debug=False
)

print(cmd)


python -m http.server --bind 127.0.0.1 --directory public --verbose


### Explanation

Many command-line tools interpret options in order.

Ordered `**kwargs` lets us write a readable builder while still preserving the exact sequence of options chosen by the caller.


---

## Problem 9: Ordered JSON Object Builder

Create a function called `json_object`.

It should accept ordered keyword arguments and return a JSON string whose keys appear in the same order.

Requirements:

1. Preserve keyword argument order.
2. Support nested dictionaries.
3. Pretty-print with an indentation of 2 spaces.
4. Do not sort keys alphabetically.


### Solution

In [19]:
def json_object(**items):
    return json.dumps(items, indent=2, sort_keys=False)


In [20]:
payload = json_object(
    id=101,
    username='alice',
    profile={
        'email': 'alice@example.com',
        'active': True
    },
    roles=['admin', 'editor']
)

print(payload)


{
  "id": 101,
  "username": "alice",
  "profile": {
    "email": "alice@example.com",
    "active": true
  },
  "roles": [
    "admin",
    "editor"
  ]
}


### Explanation

Although JSON objects are often treated conceptually as unordered mappings, preserving order is useful for human-readable API output, snapshots, test fixtures, and documentation examples.


---

## Problem 10: Create a Mini HTML Builder

Write a function called `html_tag`.

It should create an HTML element with ordered attributes.

Rules:

1. Preserve attribute order.
2. Boolean `True` means render only the attribute name.
3. Boolean `False` or `None` means omit the attribute.
4. Replace trailing underscore in attribute names, e.g. `class_='btn'` becomes `class="btn"`.
5. Replace inner underscores with hyphens, e.g. `data_id='10'` becomes `data-id="10"`.


### Solution

In [21]:
def normalize_html_attr(name):
    if name.endswith('_'):
        name = name[:-1]
    return name.replace('_', '-')


def html_tag(tag_name, **attrs):
    rendered_attrs = []

    for attr_name, value in attrs.items():
        if value is False or value is None:
            continue

        normalized_name = normalize_html_attr(attr_name)

        if value is True:
            rendered_attrs.append(normalized_name)
        else:
            rendered_attrs.append(f'{normalized_name}="{value}"')

    if rendered_attrs:
        return f'<{tag_name} ' + ' '.join(rendered_attrs) + '>'

    return f'<{tag_name}>'


In [22]:
print(html_tag('input', type='text', name='username', required=True))
print(html_tag('button', class_='btn btn-primary', data_id='save', disabled=False))


<input type="text" name="username" required>
<button class="btn btn-primary" data-id="save">


### Explanation

Attribute order can matter for readability and testing.

Ordered `**kwargs` lets a small HTML builder preserve the order chosen by the caller.


---

## Problem 11: Ordered Function Signature Generator

Create a function called `make_signature`.

It should accept ordered keyword arguments where each key is a parameter name and each value is its default value.

Requirements:

1. Preserve parameter order.
2. Use `repr()` for default values.
3. Validate parameter names as valid identifiers.
4. Reject Python keywords.


### Solution

In [23]:
def make_signature(**params):
    pieces = []

    for name, default in params.items():
        if not name.isidentifier():
            raise ValueError(f'{name!r} is not a valid parameter name.')
        if keyword.iskeyword(name):
            raise ValueError(f'{name!r} is a Python keyword.')

        pieces.append(f'{name}={default!r}')

    return '(' + ', '.join(pieces) + ')'


In [24]:
sig = make_signature(x=0, y=0, color='red', visible=True)
print(sig)


(x=0, y=0, color='red', visible=True)


### Explanation

A generated function signature must have deterministic parameter order.

Ordered `**kwargs` gives us a natural way to let users define that order directly at the call site.


---

## Problem 12: Compare Input Order vs Bound Signature Order

Write a function called `analyze_call_order`.

It should accept a function and arbitrary arguments.

It should return:

1. The order in which keyword arguments were provided by the caller.
2. The order in which arguments appear after binding to the function's signature.

Use `inspect.signature`.


### Solution

In [25]:
def analyze_call_order(func, *args, **kwargs):
    call_keyword_order = tuple(kwargs.keys())

    sig = inspect.signature(func)
    bound = sig.bind_partial(*args, **kwargs)
    bound.apply_defaults()

    bound_signature_order = tuple(bound.arguments.keys())

    return {
        'call_keyword_order': call_keyword_order,
        'bound_signature_order': bound_signature_order,
        'bound_arguments': bound.arguments
    }


In [26]:
def example(a, b=2, c=3, d=4):
    return a + b + c + d

analysis = analyze_call_order(example, c=30, a=10, d=40)

print(analysis['call_keyword_order'])
print(analysis['bound_signature_order'])
print(analysis['bound_arguments'])


('c', 'a', 'd')
('a', 'b', 'c', 'd')
{'a': 10, 'b': 2, 'c': 30, 'd': 40}


### Explanation

`kwargs.keys()` reflects the caller's keyword order.

`inspect.signature(...).bind_partial(...)` organizes arguments according to the function's signature.

This distinction is important when building decorators, validators, loggers, and tracing tools.


---

## Problem 13: Ordered Configuration Merge

Create a function called `merge_config`.

It should accept a base dictionary and ordered keyword overrides.

Requirements:

1. Existing keys keep their original position.
2. New keys are appended in the order provided by `**kwargs`.
3. Override values replace existing values.
4. Return an `OrderedDict`.


### Solution

In [27]:
def merge_config(base, **overrides):
    result = OrderedDict(base)

    for key, value in overrides.items():
        result[key] = value

    return result


In [28]:
base_config = OrderedDict([
    ('host', 'localhost'),
    ('port', 8000),
    ('debug', False)
])

merged = merge_config(
    base_config,
    debug=True,
    workers=4,
    timeout=30
)

print(merged)
print(list(merged.keys()))


OrderedDict({'host': 'localhost', 'port': 8000, 'debug': True, 'workers': 4, 'timeout': 30})
['host', 'port', 'debug', 'workers', 'timeout']


### Explanation

Assigning to an existing key updates its value without moving it in an `OrderedDict`.

New keys are appended in the order they appear in `overrides`.

This gives predictable and user-friendly configuration output.


---

## Problem 14: Ordered API Response Table

Create a function called `table_row_factory`.

It should accept ordered keyword arguments where keys are column names and values are default values.

The function should return another function called `make_row`.

Requirements:

1. Column order is defined by the original factory call.
2. Missing values use defaults.
3. Unexpected override keys are rejected.
4. Returned rows follow column order, not override order.


### Solution

In [29]:
def table_row_factory(**columns):
    column_names = tuple(columns.keys())
    defaults = dict(columns)

    def make_row(**overrides):
        unexpected = set(overrides) - set(column_names)
        if unexpected:
            ordered_unexpected = [name for name in overrides if name in unexpected]
            raise ValueError(f'Unexpected column(s): {ordered_unexpected}')

        row = OrderedDict()
        for column_name in column_names:
            row[column_name] = overrides.get(column_name, defaults[column_name])

        return row

    make_row.columns = column_names
    return make_row


In [30]:
make_user_row = table_row_factory(
    id=None,
    username='',
    email='',
    active=True
)

row = make_user_row(active=False, username='alice', id=1)

print(row)
print(make_user_row.columns)


OrderedDict({'id': 1, 'username': 'alice', 'email': '', 'active': False})
('id', 'username', 'email', 'active')


### Explanation

The factory captures the original `**columns` order.

Each generated row follows that captured order, regardless of the order in which overrides are passed later.


---

## Problem 15: Ordered Mini Data Class Factory

Create a function called `mini_dataclass`.

It should accept a class name and ordered keyword arguments representing field defaults.

It should dynamically create a simple class with:

1. An `__init__` method using the provided defaults.
2. A `__repr__` method that displays fields in the original order.
3. A `to_dict` method that returns an `OrderedDict` in field order.
4. A class-level `__fields__` attribute containing field names in order.

Do not use the `dataclasses` module.


### Solution

In [31]:
def mini_dataclass(class_name, **field_defaults):
    if not field_defaults:
        raise ValueError('At least one field is required.')

    field_names = tuple(field_defaults.keys())

    def __init__(self, **overrides):
        unexpected = set(overrides) - set(field_names)
        if unexpected:
            ordered_unexpected = [name for name in overrides if name in unexpected]
            raise ValueError(f'Unexpected field(s): {ordered_unexpected}')

        for field_name in field_names:
            value = overrides.get(field_name, field_defaults[field_name])
            setattr(self, field_name, value)

    def __repr__(self):
        pieces = []
        for field_name in field_names:
            pieces.append(f'{field_name}={getattr(self, field_name)!r}')
        return f'{class_name}(' + ', '.join(pieces) + ')'

    def to_dict(self):
        return OrderedDict(
            (field_name, getattr(self, field_name))
            for field_name in field_names
        )

    namespace = {
        '__fields__': field_names,
        '__init__': __init__,
        '__repr__': __repr__,
        'to_dict': to_dict
    }

    return type(class_name, (), namespace)


In [32]:
Product = mini_dataclass(
    'Product',
    id=None,
    name='',
    price=0.0,
    in_stock=True
)

product = Product(id=501, name='Keyboard', price=99.99)

print(Product.__fields__)
print(product)
print(product.to_dict())


('id', 'name', 'price', 'in_stock')
Product(id=501, name='Keyboard', price=99.99, in_stock=True)
OrderedDict({'id': 501, 'name': 'Keyboard', 'price': 99.99, 'in_stock': True})


### Explanation

The generated class depends on ordered `**kwargs` in several places:

- field declaration order
- representation order
- dictionary export order
- predictable initialization behavior

This is similar in spirit to why modern class bodies and dataclass fields also care about definition order.


---

## Problem 16: Advanced Challenge — Ordered Query Language Builder

Create a function called `select`.

It should accept ordered keyword arguments where each key is an output alias and each value is a source expression.

Requirements:

1. Preserve selected column order.
2. Validate aliases as identifiers.
3. Validate that expressions are non-empty strings.
4. Return the SQL string.
5. Raise clear errors in user-provided order.


### Solution

In [33]:
def select(**expressions):
    if not expressions:
        raise ValueError('At least one expression is required.')

    parts = []

    for alias, expression in expressions.items():
        if not alias.isidentifier():
            raise ValueError(f'Alias {alias!r} is not a valid identifier.')
        if keyword.iskeyword(alias):
            raise ValueError(f'Alias {alias!r} is a Python keyword.')
        if not isinstance(expression, str) or not expression.strip():
            raise ValueError(f'Expression for alias {alias!r} must be a non-empty string.')

        parts.append(f'{expression} AS {alias}')

    return 'SELECT ' + ', '.join(parts)


In [34]:
query = select(
    user_id='users.id',
    username='users.name',
    order_count='COUNT(orders.id)',
    total_spent='SUM(orders.total)'
)

print(query)


SELECT users.id AS user_id, users.name AS username, COUNT(orders.id) AS order_count, SUM(orders.total) AS total_spent


### Explanation

The order of selected expressions controls the order of columns in the query result.

Ordered `**kwargs` lets the API stay compact and expressive while keeping output deterministic.


---

# Summary

Ordered `**kwargs` enables cleaner APIs for problems where order is meaningful.

Common use cases include:

- named tuple factories
- schema builders
- SQL builders
- CLI builders
- HTML builders
- JSON serialization
- validation systems
- decorators and tracing tools
- mini domain-specific languages

The main best practice is simple:

Use ordered `**kwargs` when the call-site order improves readability and when preserving that order makes the resulting behavior clearer and more predictable.
